# Optimizacion y Evaluacion de Prompts

## Objetivo
Aprender a evaluar y optimizar prompts de manera sistematica, utilizando metricas cuantitativas
y tecnicas de mejora iterativa.

## Ciclo de Optimizacion de Prompts

La optimizacion de prompts sigue un ciclo iterativo:

1. **Disenar** un prompt inicial basado en la tarea
2. **Ejecutar** el prompt multiples veces para obtener resultados
3. **Medir** la calidad de las respuestas usando metricas definidas
4. **Analizar** los resultados para identificar debilidades
5. **Mejorar** el prompt basandose en el analisis
6. **Repetir** hasta alcanzar la calidad deseada

Este enfoque sistematico nos permite pasar de prompts intuitivos a prompts
optimizados con evidencia cuantitativa de su efectividad.

## Contenido
1. Metricas de Evaluacion de Prompts
2. A/B Testing de Prompts
3. Prompt Templates y Variables
4. Optimizacion Iterativa
5. Ejercicio Practico

## Configuracion del Entorno

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verificar credenciales
if not os.getenv("GITHUB_TOKEN"):
    raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")
print("Entorno configurado correctamente")

Entorno configurado correctamente


In [2]:
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL", "https://models.inference.ai.azure.com"),
    api_key=os.getenv("GITHUB_TOKEN")
)

MODELO = "gpt-4o-mini"
print(f"Cliente configurado con modelo: {MODELO}")

Cliente configurado con modelo: gpt-4o-mini


## Funcion auxiliar para llamadas al LLM

In [3]:
def llamar_llm(prompt, system_prompt="Eres un asistente util.", temperatura=0.7, max_tokens=500):
    """Realiza una llamada al LLM y retorna la respuesta como texto."""
    respuesta = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=temperatura,
        max_tokens=max_tokens
    )
    return respuesta.choices[0].message.content

---
## Seccion 1: Metricas de Evaluacion de Prompts

Para optimizar prompts necesitamos metricas cuantitativas que nos permitan comparar
diferentes versiones de un prompt. Las metricas principales son:

- **Consistencia**: Si ejecutamos el mismo prompt N veces, que tan similares son las respuestas?
- **Longitud de respuesta**: La respuesta tiene la extension esperada?
- **Cumplimiento de formato**: La respuesta sigue el formato solicitado?

Estas metricas nos dan una base objetiva para comparar prompts.

### 1.1 Metrica de Consistencia

Ejecutamos el mismo prompt multiples veces y medimos que tan similares son las respuestas.
Usamos similitud de Jaccard entre pares de respuestas como medida simple.

In [4]:
def similitud_jaccard(texto1, texto2):
    """Calcula la similitud de Jaccard entre dos textos basada en palabras."""
    palabras1 = set(texto1.lower().split())
    palabras2 = set(texto2.lower().split())
    if not palabras1 or not palabras2:
        return 0.0
    interseccion = palabras1 & palabras2
    union = palabras1 | palabras2
    return len(interseccion) / len(union)


def medir_consistencia(prompt, n_ejecuciones=3, temperatura=0.7):
    """Ejecuta un prompt N veces y mide la consistencia entre respuestas."""
    respuestas = []
    for i in range(n_ejecuciones):
        respuesta = llamar_llm(prompt, temperatura=temperatura)
        respuestas.append(respuesta)
        print(f"  Ejecucion {i+1}: {respuesta[:80]}...")
    
    # Calcular similitud promedio entre todos los pares
    similitudes = []
    for i in range(len(respuestas)):
        for j in range(i+1, len(respuestas)):
            sim = similitud_jaccard(respuestas[i], respuestas[j])
            similitudes.append(sim)
    
    consistencia_promedio = sum(similitudes) / len(similitudes) if similitudes else 0
    return {
        "respuestas": respuestas,
        "consistencia_promedio": consistencia_promedio,
        "similitudes_pares": similitudes
    }


# Ejemplo: comparar consistencia con diferentes temperaturas
prompt_test = "Nombra 3 ventajas de usar Python para ciencia de datos."

print("=== Temperatura 0.0 (determinista) ===")
resultado_baja = medir_consistencia(prompt_test, n_ejecuciones=3, temperatura=0.0)
print(f"Consistencia promedio: {resultado_baja['consistencia_promedio']:.3f}\n")

print("=== Temperatura 1.0 (creativa) ===")
resultado_alta = medir_consistencia(prompt_test, n_ejecuciones=3, temperatura=1.0)
print(f"Consistencia promedio: {resultado_alta['consistencia_promedio']:.3f}")

=== Temperatura 0.0 (determinista) ===
  Ejecucion 1: Claro, aquí tienes tres ventajas de usar Python para ciencia de datos:

1. **Bib...
  Ejecucion 2: Claro, aquí tienes tres ventajas de usar Python para ciencia de datos:

1. **Bib...
  Ejecucion 3: Claro, aquí tienes tres ventajas de usar Python para ciencia de datos:

1. **Bib...
Consistencia promedio: 0.776

=== Temperatura 1.0 (creativa) ===
  Ejecucion 1: Claro, aquí tienes tres ventajas de usar Python para ciencia de datos:

1. **Bib...
  Ejecucion 2: Claro, aquí tienes tres ventajas de usar Python para ciencia de datos:

1. **Bib...
  Ejecucion 3: Claro, aquí tienes tres ventajas de usar Python para la ciencia de datos:

1. **...
Consistencia promedio: 0.426


### 1.2 Metrica de Longitud de Respuesta

In [5]:
def medir_longitud(prompt, longitud_objetivo, tolerancia=0.3):
    """Mide si la respuesta cumple con la longitud objetivo.
    
    Args:
        prompt: El prompt a evaluar
        longitud_objetivo: Numero de palabras esperado
        tolerancia: Porcentaje de desviacion aceptable (0.3 = 30%)
    
    Returns:
        dict con longitud real, objetivo, y si cumple la tolerancia
    """
    respuesta = llamar_llm(prompt, temperatura=0.3)
    palabras = len(respuesta.split())
    
    desviacion = abs(palabras - longitud_objetivo) / longitud_objetivo
    cumple = desviacion <= tolerancia
    
    return {
        "respuesta": respuesta,
        "palabras_reales": palabras,
        "palabras_objetivo": longitud_objetivo,
        "desviacion_porcentual": desviacion * 100,
        "cumple_tolerancia": cumple
    }


# Probar con un prompt que pide respuesta corta
prompt_corto = "En exactamente 50 palabras, explica que es machine learning."
resultado = medir_longitud(prompt_corto, longitud_objetivo=50)

print(f"Palabras objetivo: {resultado['palabras_objetivo']}")
print(f"Palabras reales: {resultado['palabras_reales']}")
print(f"Desviacion: {resultado['desviacion_porcentual']:.1f}%")
print(f"Cumple tolerancia (30%): {resultado['cumple_tolerancia']}")
print(f"\nRespuesta: {resultado['respuesta']}")

Palabras objetivo: 50
Palabras reales: 50
Desviacion: 0.0%
Cumple tolerancia (30%): True

Respuesta: El machine learning, o aprendizaje automático, es una rama de la inteligencia artificial que permite a las computadoras aprender de datos y mejorar su rendimiento en tareas específicas sin ser programadas explícitamente. Utiliza algoritmos para identificar patrones, hacer predicciones y tomar decisiones basadas en datos, transformando diversas industrias y aplicaciones.


### 1.3 Metrica de Cumplimiento de Formato

Verificamos si la respuesta sigue un formato especifico solicitado en el prompt.

In [6]:
import json


def verificar_formato_json(prompt):
    """Verifica si la respuesta es JSON valido."""
    respuesta = llamar_llm(prompt, temperatura=0.0)
    
    # Intentar extraer JSON de la respuesta
    texto_limpio = respuesta.strip()
    # Remover bloques de codigo markdown si existen
    if texto_limpio.startswith("```"):
        lineas = texto_limpio.split("\n")
        texto_limpio = "\n".join(lineas[1:-1])
    
    try:
        datos = json.loads(texto_limpio)
        return {"es_json_valido": True, "datos": datos, "respuesta_raw": respuesta}
    except json.JSONDecodeError as e:
        return {"es_json_valido": False, "error": str(e), "respuesta_raw": respuesta}


def verificar_formato_lista(prompt, min_items=3):
    """Verifica si la respuesta contiene una lista numerada con minimo N items."""
    respuesta = llamar_llm(prompt, temperatura=0.0)
    
    import re
    # Buscar patrones de lista numerada: 1. , 2. , etc.
    items = re.findall(r'^\d+\.\s+.+', respuesta, re.MULTILINE)
    
    return {
        "cumple_formato": len(items) >= min_items,
        "items_encontrados": len(items),
        "items_requeridos": min_items,
        "items": items,
        "respuesta_raw": respuesta
    }


# Probar formato JSON
prompt_json = """Describe 3 lenguajes de programacion populares.
Responde UNICAMENTE con un JSON valido con esta estructura:
[{"nombre": "...", "uso_principal": "...", "anio_creacion": ...}]"""

resultado_json = verificar_formato_json(prompt_json)
print(f"JSON valido: {resultado_json['es_json_valido']}")
if resultado_json['es_json_valido']:
    print(f"Datos parseados: {json.dumps(resultado_json['datos'], indent=2, ensure_ascii=False)}")
else:
    print(f"Error: {resultado_json['error']}")
    print(f"Respuesta raw: {resultado_json['respuesta_raw']}")

print("\n" + "="*50 + "\n")

# Probar formato lista
prompt_lista = "Lista 5 frameworks de desarrollo web populares."
resultado_lista = verificar_formato_lista(prompt_lista, min_items=5)
print(f"Cumple formato lista: {resultado_lista['cumple_formato']}")
print(f"Items encontrados: {resultado_lista['items_encontrados']} / {resultado_lista['items_requeridos']}")
for item in resultado_lista['items']:
    print(f"  {item}")

JSON valido: True
Datos parseados: [
  {
    "nombre": "Python",
    "uso_principal": "Desarrollo web, análisis de datos, inteligencia artificial, automatización de tareas.",
    "anio_creacion": 1991
  },
  {
    "nombre": "JavaScript",
    "uso_principal": "Desarrollo web, programación del lado del cliente y del servidor, aplicaciones móviles.",
    "anio_creacion": 1995
  },
  {
    "nombre": "Java",
    "uso_principal": "Desarrollo de aplicaciones empresariales, aplicaciones móviles (Android), sistemas embebidos.",
    "anio_creacion": 1995
  }
]


Cumple formato lista: True
Items encontrados: 5 / 5
  1. **React**: Una biblioteca de JavaScript para construir interfaces de usuario, mantenida por Facebook. Es ampliamente utilizada para desarrollar aplicaciones de una sola página (SPA).
  2. **Angular**: Un framework de desarrollo web de código abierto mantenido por Google. Es ideal para crear aplicaciones web dinámicas y de una sola página.
  3. **Vue.js**: Un framework progresivo de

---
## Seccion 2: A/B Testing de Prompts

El A/B testing nos permite comparar dos versiones de un prompt de manera objetiva.
Definimos una tarea, creamos dos variantes del prompt, y usamos metricas
automatizadas para determinar cual es mejor.

### Metodologia
1. Definir la tarea y los criterios de exito
2. Crear dos variantes del prompt (A y B)
3. Ejecutar ambas variantes multiples veces
4. Evaluar las respuestas con metricas automatizadas
5. Comparar resultados y seleccionar el ganador

In [ ]:
def evaluar_calidad_respuesta(pregunta, respuesta, criterios):
    """Usa el LLM como juez para evaluar la calidad de una respuesta.
    
    Args:
        pregunta: La pregunta original
        respuesta: La respuesta a evaluar
        criterios: Lista de criterios de evaluacion
    
    Returns:
        dict con puntaje y justificacion
    """
    criterios_texto = "\n".join([f"- {c}" for c in criterios])
    
    prompt_evaluacion = f"""Evalua la siguiente respuesta a la pregunta dada.

PREGUNTA: {pregunta}

RESPUESTA A EVALUAR:
{respuesta}

CRITERIOS DE EVALUACION:
{criterios_texto}

Responde UNICAMENTE con un JSON valido con esta estructura:
{{"puntaje": <numero del 1 al 10>, "justificacion": "<breve explicacion>"}}"""
    
    resultado = llamar_llm(prompt_evaluacion, temperatura=0.0)
    
    # Parsear resultado
    texto_limpio = resultado.strip()
    if texto_limpio.startswith("```"):
        lineas = texto_limpio.split("\n")
        texto_limpio = "\n".join(lineas[1:-1])
    
    try:
        return json.loads(texto_limpio)
    except json.JSONDecodeError:
        return {"puntaje": 0, "justificacion": f"Error al parsear evaluacion: {resultado}"}

In [ ]:
def ab_test(prompt_a, prompt_b, nombre_a, nombre_b, preguntas_test, criterios, n_repeticiones=2):
    """Realiza un A/B test comparando dos prompts.
    
    Args:
        prompt_a: Template del prompt A (debe contener {pregunta})
        prompt_b: Template del prompt B (debe contener {pregunta})
        nombre_a: Nombre descriptivo del prompt A
        nombre_b: Nombre descriptivo del prompt B
        preguntas_test: Lista de preguntas para probar
        criterios: Criterios de evaluacion
        n_repeticiones: Veces que se repite cada pregunta
    
    Returns:
        dict con resultados del A/B test
    """
    resultados_a = []
    resultados_b = []
    
    for pregunta in preguntas_test:
        print(f"\nProbando con: '{pregunta[:60]}...'")
        
        for rep in range(n_repeticiones):
            # Ejecutar prompt A
            respuesta_a = llamar_llm(prompt_a.format(pregunta=pregunta), temperatura=0.3)
            eval_a = evaluar_calidad_respuesta(pregunta, respuesta_a, criterios)
            resultados_a.append(eval_a["puntaje"])
            
            # Ejecutar prompt B
            respuesta_b = llamar_llm(prompt_b.format(pregunta=pregunta), temperatura=0.3)
            eval_b = evaluar_calidad_respuesta(pregunta, respuesta_b, criterios)
            resultados_b.append(eval_b["puntaje"])
            
            print(f"  Rep {rep+1}: {nombre_a}={eval_a['puntaje']}, {nombre_b}={eval_b['puntaje']}")
    
    promedio_a = sum(resultados_a) / len(resultados_a) if resultados_a else 0
    promedio_b = sum(resultados_b) / len(resultados_b) if resultados_b else 0
    
    return {
        "nombre_a": nombre_a,
        "nombre_b": nombre_b,
        "promedio_a": promedio_a,
        "promedio_b": promedio_b,
        "puntajes_a": resultados_a,
        "puntajes_b": resultados_b,
        "ganador": nombre_a if promedio_a > promedio_b else nombre_b
    }

In [ ]:
# Definir dos variantes de prompt para explicar conceptos tecnicos
prompt_basico = "{pregunta}"

prompt_estructurado = """Responde la siguiente pregunta de forma clara y estructurada.
Usa el siguiente formato:
- Definicion breve (1-2 oraciones)
- Ejemplo practico
- Analogia simple para entender el concepto

Pregunta: {pregunta}"""

# Preguntas de prueba
preguntas = [
    "Que es una API REST?",
    "Que es el aprendizaje por refuerzo?",
    "Que es Docker?"
]

# Criterios de evaluacion
criterios = [
    "Claridad: La explicacion es facil de entender",
    "Completitud: Cubre los aspectos esenciales del concepto",
    "Estructura: La respuesta esta bien organizada",
    "Ejemplo: Incluye un ejemplo practico o analogia"
]

# Ejecutar A/B test
resultados = ab_test(
    prompt_a=prompt_basico,
    prompt_b=prompt_estructurado,
    nombre_a="Basico",
    nombre_b="Estructurado",
    preguntas_test=preguntas,
    criterios=criterios,
    n_repeticiones=1
)

print("\n" + "="*50)
print("RESULTADOS DEL A/B TEST")
print("="*50)
print(f"Prompt '{resultados['nombre_a']}': promedio = {resultados['promedio_a']:.2f}")
print(f"Prompt '{resultados['nombre_b']}': promedio = {resultados['promedio_b']:.2f}")
print(f"\nGanador: {resultados['ganador']}")

---
## Seccion 3: Prompt Templates y Variables

Los prompt templates son plantillas reutilizables que nos permiten:
- **Estandarizar** la estructura de nuestros prompts
- **Parametrizar** variables como el tono, idioma, nivel de detalle
- **Reutilizar** prompts probados en diferentes contextos
- **Versionar** y comparar diferentes iteraciones

In [ ]:
class PromptTemplate:
    """Clase para crear y gestionar prompt templates parametrizables."""
    
    def __init__(self, nombre, template, variables_default=None):
        self.nombre = nombre
        self.template = template
        self.variables_default = variables_default or {}
        self.version = 1
        self.historial = []
    
    def render(self, **kwargs):
        """Renderiza el template con las variables proporcionadas."""
        variables = {**self.variables_default, **kwargs}
        try:
            return self.template.format(**variables)
        except KeyError as e:
            raise ValueError(f"Variable faltante en template '{self.nombre}': {e}")
    
    def actualizar(self, nuevo_template):
        """Actualiza el template guardando el historial."""
        self.historial.append({
            "version": self.version,
            "template": self.template
        })
        self.template = nuevo_template
        self.version += 1
    
    def ejecutar(self, client_fn, **kwargs):
        """Renderiza y ejecuta el prompt."""
        prompt_renderizado = self.render(**kwargs)
        return client_fn(prompt_renderizado)
    
    def __repr__(self):
        return f"PromptTemplate('{self.nombre}', v{self.version})"

In [ ]:
# Crear un template para resumen de textos
template_resumen = PromptTemplate(
    nombre="resumen_texto",
    template="""Resume el siguiente texto en {num_oraciones} oraciones.
Nivel de detalle: {nivel_detalle}
Tono: {tono}
Idioma de salida: {idioma}

TEXTO:
{texto}

RESUMEN:""",
    variables_default={
        "num_oraciones": 3,
        "nivel_detalle": "intermedio",
        "tono": "profesional",
        "idioma": "espanol"
    }
)

# Texto de ejemplo para resumir
texto_ejemplo = """La inteligencia artificial (IA) ha experimentado un crecimiento exponencial 
en los ultimos anos, impulsada principalmente por avances en deep learning y la disponibilidad 
de grandes volumenes de datos. Los modelos de lenguaje grandes (LLMs) como GPT-4 han 
demostrado capacidades impresionantes en generacion de texto, traduccion, y razonamiento. 
Sin embargo, estos modelos tambien presentan desafios significativos, incluyendo 
alucinaciones, sesgos en los datos de entrenamiento, y altos costos computacionales. 
La comunidad cientifica trabaja activamente en soluciones como RAG (Retrieval-Augmented 
Generation), fine-tuning eficiente con LoRA, y tecnicas de alineamiento con valores humanos."""

# Usar con valores por defecto
print("=== Resumen con valores por defecto ===")
prompt_default = template_resumen.render(texto=texto_ejemplo)
respuesta = llamar_llm(prompt_default, temperatura=0.3)
print(respuesta)

print("\n=== Resumen corto y tecnico ===")
prompt_tecnico = template_resumen.render(
    texto=texto_ejemplo,
    num_oraciones=2,
    nivel_detalle="alto",
    tono="tecnico"
)
respuesta_tecnica = llamar_llm(prompt_tecnico, temperatura=0.3)
print(respuesta_tecnica)

print("\n=== Resumen simple e informal ===")
prompt_simple = template_resumen.render(
    texto=texto_ejemplo,
    num_oraciones=2,
    nivel_detalle="bajo",
    tono="informal y amigable"
)
respuesta_simple = llamar_llm(prompt_simple, temperatura=0.3)
print(respuesta_simple)

In [ ]:
# Template para clasificacion de texto
template_clasificacion = PromptTemplate(
    nombre="clasificar_texto",
    template="""Clasifica el siguiente texto en una de estas categorias: {categorias}

Texto: "{texto}"

Responde UNICAMENTE con un JSON valido:
{{"categoria": "<categoria elegida>", "confianza": <0.0 a 1.0>, "razon": "<breve explicacion>"}}""",
    variables_default={
        "categorias": "positivo, negativo, neutro"
    }
)

# Probar con diferentes textos
textos_prueba = [
    "El nuevo producto es increible, supero todas mis expectativas!",
    "El servicio fue pesimo, nunca volvere a comprar aqui.",
    "Recibi el paquete ayer por la tarde."
]

print("=== Clasificacion de Sentimiento ===\n")
for texto in textos_prueba:
    prompt = template_clasificacion.render(texto=texto)
    resultado = llamar_llm(prompt, temperatura=0.0)
    print(f"Texto: '{texto[:50]}...'")
    print(f"Resultado: {resultado}\n")

---
## Seccion 4: Optimizacion Iterativa

En esta seccion aplicamos el ciclo completo de optimizacion:
1. Comenzamos con un prompt basico
2. Medimos su rendimiento
3. Identificamos problemas
4. Mejoramos el prompt
5. Medimos nuevamente y comparamos

### Tarea: Extraccion de informacion estructurada
Queremos extraer datos estructurados de descripciones de productos.

In [ ]:
# Datos de prueba: descripciones de productos
descripciones_productos = [
    "Laptop Dell XPS 15, procesador Intel i7 de 12va generacion, 16GB RAM DDR5, SSD de 512GB, pantalla OLED 15.6 pulgadas, precio $1,299 USD",
    "Telefono Samsung Galaxy S24 Ultra con 256GB de almacenamiento, 12GB RAM, camara de 200MP, bateria de 5000mAh, disponible por $1,199",
    "Monitor LG UltraWide 34 pulgadas, resolucion 3440x1440, panel IPS, 144Hz, HDR10, USB-C, precio de venta: $449.99"
]

# Estructura esperada
estructura_esperada = {
    "nombre": "str",
    "marca": "str", 
    "categoria": "str",
    "precio_usd": "float",
    "especificaciones": ["str"]
}

print("Estructura esperada:")
print(json.dumps(estructura_esperada, indent=2))

In [ ]:
def evaluar_extraccion(respuesta_raw, campos_requeridos):
    """Evalua la calidad de una extraccion de datos estructurados."""
    resultados = {
        "es_json_valido": False,
        "campos_presentes": [],
        "campos_faltantes": [],
        "puntaje": 0
    }
    
    # Intentar parsear JSON
    texto_limpio = respuesta_raw.strip()
    if texto_limpio.startswith("```"):
        lineas = texto_limpio.split("\n")
        texto_limpio = "\n".join(lineas[1:-1])
    
    try:
        datos = json.loads(texto_limpio)
        resultados["es_json_valido"] = True
    except json.JSONDecodeError:
        resultados["puntaje"] = 0
        return resultados
    
    # Verificar campos requeridos
    for campo in campos_requeridos:
        if campo in datos and datos[campo] is not None and datos[campo] != "":
            resultados["campos_presentes"].append(campo)
        else:
            resultados["campos_faltantes"].append(campo)
    
    # Calcular puntaje
    total_campos = len(campos_requeridos)
    campos_ok = len(resultados["campos_presentes"])
    # JSON valido vale 20%, campos valen 80%
    resultados["puntaje"] = 20 + (80 * campos_ok / total_campos)
    
    return resultados

In [ ]:
# === ITERACION 1: Prompt basico ===
prompt_v1 = """Extrae los datos del siguiente producto y devuelvelos como JSON:
{descripcion}"""

campos_requeridos = ["nombre", "marca", "categoria", "precio_usd", "especificaciones"]

print("=" * 60)
print("ITERACION 1: Prompt Basico")
print("=" * 60)

puntajes_v1 = []
for desc in descripciones_productos:
    respuesta = llamar_llm(prompt_v1.format(descripcion=desc), temperatura=0.0)
    evaluacion = evaluar_extraccion(respuesta, campos_requeridos)
    puntajes_v1.append(evaluacion["puntaje"])
    print(f"\nProducto: {desc[:50]}...")
    print(f"  JSON valido: {evaluacion['es_json_valido']}")
    print(f"  Campos presentes: {evaluacion['campos_presentes']}")
    print(f"  Campos faltantes: {evaluacion['campos_faltantes']}")
    print(f"  Puntaje: {evaluacion['puntaje']:.0f}/100")

promedio_v1 = sum(puntajes_v1) / len(puntajes_v1)
print(f"\nPuntaje promedio V1: {promedio_v1:.1f}/100")

In [ ]:
# === ITERACION 2: Prompt mejorado con estructura explicita ===
prompt_v2 = """Extrae los datos estructurados del siguiente producto.

Responde UNICAMENTE con un JSON valido (sin texto adicional ni bloques de codigo) con esta estructura exacta:
{{
    "nombre": "nombre completo del producto",
    "marca": "marca del fabricante",
    "categoria": "laptop|telefono|monitor|tablet|otro",
    "precio_usd": 0.00,
    "especificaciones": ["spec1", "spec2", "spec3"]
}}

Reglas:
- precio_usd debe ser un numero decimal (sin simbolo $)
- especificaciones debe ser una lista de strings con las caracteristicas tecnicas principales
- Si algun dato no esta disponible, usa null

DESCRIPCION DEL PRODUCTO:
{descripcion}"""

print("=" * 60)
print("ITERACION 2: Prompt con estructura explicita")
print("=" * 60)

puntajes_v2 = []
for desc in descripciones_productos:
    respuesta = llamar_llm(prompt_v2.format(descripcion=desc), temperatura=0.0)
    evaluacion = evaluar_extraccion(respuesta, campos_requeridos)
    puntajes_v2.append(evaluacion["puntaje"])
    print(f"\nProducto: {desc[:50]}...")
    print(f"  JSON valido: {evaluacion['es_json_valido']}")
    print(f"  Campos presentes: {evaluacion['campos_presentes']}")
    print(f"  Campos faltantes: {evaluacion['campos_faltantes']}")
    print(f"  Puntaje: {evaluacion['puntaje']:.0f}/100")

promedio_v2 = sum(puntajes_v2) / len(puntajes_v2)
print(f"\nPuntaje promedio V2: {promedio_v2:.1f}/100")

In [ ]:
# === ITERACION 3: Prompt con ejemplo (few-shot) ===
prompt_v3 = """Extrae los datos estructurados del siguiente producto.

EJEMPLO:
Descripcion: "Auriculares Sony WH-1000XM5, cancelacion de ruido activa, Bluetooth 5.3, bateria 30 horas, precio $349"
Respuesta:
{{"nombre": "WH-1000XM5", "marca": "Sony", "categoria": "auriculares", "precio_usd": 349.00, "especificaciones": ["Cancelacion de ruido activa", "Bluetooth 5.3", "Bateria 30 horas"]}}

INSTRUCCIONES:
- Responde UNICAMENTE con JSON valido, sin texto adicional
- precio_usd es un numero decimal sin simbolo $
- especificaciones es una lista de las caracteristicas tecnicas clave
- categoria puede ser: laptop, telefono, monitor, tablet, auriculares, otro

DESCRIPCION DEL PRODUCTO:
{descripcion}"""

print("=" * 60)
print("ITERACION 3: Prompt con ejemplo (few-shot)")
print("=" * 60)

puntajes_v3 = []
for desc in descripciones_productos:
    respuesta = llamar_llm(prompt_v3.format(descripcion=desc), temperatura=0.0)
    evaluacion = evaluar_extraccion(respuesta, campos_requeridos)
    puntajes_v3.append(evaluacion["puntaje"])
    print(f"\nProducto: {desc[:50]}...")
    print(f"  JSON valido: {evaluacion['es_json_valido']}")
    print(f"  Campos presentes: {evaluacion['campos_presentes']}")
    print(f"  Campos faltantes: {evaluacion['campos_faltantes']}")
    print(f"  Puntaje: {evaluacion['puntaje']:.0f}/100")

promedio_v3 = sum(puntajes_v3) / len(puntajes_v3)
print(f"\nPuntaje promedio V3: {promedio_v3:.1f}/100")

In [ ]:
# === Resumen de la optimizacion iterativa ===
print("=" * 60)
print("RESUMEN DE OPTIMIZACION ITERATIVA")
print("=" * 60)
print(f"")
print(f"V1 (Basico):              {promedio_v1:.1f}/100")
print(f"V2 (Estructura explicita): {promedio_v2:.1f}/100")
print(f"V3 (Few-shot):            {promedio_v3:.1f}/100")
print(f"")
print(f"Mejora V1 -> V2: +{promedio_v2 - promedio_v1:.1f} puntos")
print(f"Mejora V2 -> V3: +{promedio_v3 - promedio_v2:.1f} puntos")
print(f"Mejora total V1 -> V3: +{promedio_v3 - promedio_v1:.1f} puntos")

# Determinar mejor version
mejor = max([(promedio_v1, "V1"), (promedio_v2, "V2"), (promedio_v3, "V3")])
print(f"\nMejor version: {mejor[1]} con {mejor[0]:.1f}/100")

---
## Seccion 5: Ejercicio Practico

### Tarea: Optimizar un prompt para extraccion de eventos

**Objetivo**: Dado un texto en lenguaje natural que describe un evento, extraer la informacion
estructurada: nombre del evento, fecha, lugar, organizador y tipo de evento.

**Instrucciones**:
1. Comienza con el prompt basico proporcionado
2. Ejecutalo con los textos de prueba y mide el puntaje
3. Mejora el prompt iterativamente (al menos 2 versiones)
4. Documenta que cambios hiciste y por que
5. Compara los puntajes entre versiones

In [ ]:
# Textos de prueba para el ejercicio
textos_eventos = [
    "El proximo 15 de marzo de 2026 se realizara la Conferencia Internacional de IA en el Centro de Convenciones de Santiago, organizada por la Universidad de Chile. Es un evento academico abierto al publico.",
    "Hackathon de Datos Abiertos: 22-23 de abril, 2026. Lugar: Hub Providencia, Santiago. Organiza: Fundacion Datos Chile. Evento de tipo competencia tecnologica con premios.",
    "Workshop de Machine Learning con Python, dictado por DataScience SpA, sera el 5 de mayo en el auditorio principal de DUOC UC sede San Carlos de Apoquindo. Capacitacion practica de un dia."
]

campos_evento = ["nombre_evento", "fecha", "lugar", "organizador", "tipo_evento"]

# Prompt basico inicial (los estudiantes deben mejorar este prompt)
prompt_ejercicio_v1 = """Extrae la informacion del siguiente evento como JSON:
{texto}"""

print("=== Evaluacion del prompt basico ===")
puntajes_ejercicio = []
for texto in textos_eventos:
    respuesta = llamar_llm(prompt_ejercicio_v1.format(texto=texto), temperatura=0.0)
    evaluacion = evaluar_extraccion(respuesta, campos_evento)
    puntajes_ejercicio.append(evaluacion["puntaje"])
    print(f"\nEvento: {texto[:60]}...")
    print(f"  Puntaje: {evaluacion['puntaje']:.0f}/100")
    print(f"  Campos OK: {evaluacion['campos_presentes']}")
    print(f"  Campos faltantes: {evaluacion['campos_faltantes']}")

print(f"\nPuntaje promedio V1: {sum(puntajes_ejercicio)/len(puntajes_ejercicio):.1f}/100")

In [ ]:
# === ESPACIO PARA EL ESTUDIANTE ===
# Mejora el prompt y ejecuta la evaluacion nuevamente.
# Documenta que cambios realizaste y por que.

# TU PROMPT MEJORADO (Version 2):
prompt_ejercicio_v2 = """[ESCRIBE TU PROMPT MEJORADO AQUI]

{texto}"""

# Descomenta las siguientes lineas cuando tengas tu prompt listo:
# print("=== Evaluacion del prompt mejorado V2 ===")
# puntajes_v2 = []
# for texto in textos_eventos:
#     respuesta = llamar_llm(prompt_ejercicio_v2.format(texto=texto), temperatura=0.0)
#     evaluacion = evaluar_extraccion(respuesta, campos_evento)
#     puntajes_v2.append(evaluacion["puntaje"])
#     print(f"\nEvento: {texto[:60]}...")
#     print(f"  Puntaje: {evaluacion['puntaje']:.0f}/100")
# print(f"\nPuntaje promedio V2: {sum(puntajes_v2)/len(puntajes_v2):.1f}/100")

In [ ]:
# === ESPACIO PARA EL ESTUDIANTE ===
# Version 3: Aplica lo aprendido (few-shot, estructura explicita, etc.)

# TU PROMPT MEJORADO (Version 3):
prompt_ejercicio_v3 = """[ESCRIBE TU PROMPT MEJORADO AQUI]

{texto}"""

# Descomenta las siguientes lineas cuando tengas tu prompt listo:
# print("=== Evaluacion del prompt mejorado V3 ===")
# puntajes_v3 = []
# for texto in textos_eventos:
#     respuesta = llamar_llm(prompt_ejercicio_v3.format(texto=texto), temperatura=0.0)
#     evaluacion = evaluar_extraccion(respuesta, campos_evento)
#     puntajes_v3.append(evaluacion["puntaje"])
#     print(f"\nEvento: {texto[:60]}...")
#     print(f"  Puntaje: {evaluacion['puntaje']:.0f}/100")
# print(f"\nPuntaje promedio V3: {sum(puntajes_v3)/len(puntajes_v3):.1f}/100")

### Reflexion del ejercicio

Responde las siguientes preguntas:

1. **Que cambios tuvieron mayor impacto en el puntaje?**
   - (Tu respuesta aqui)

2. **Que patron de optimizacion fue mas efectivo? (estructura explicita, few-shot, reglas, etc.)**
   - (Tu respuesta aqui)

3. **Hay un limite en la mejora que se puede lograr solo con prompt engineering?**
   - (Tu respuesta aqui)

4. **Como aplicarias este enfoque de optimizacion en un proyecto real?**
   - (Tu respuesta aqui)

---
## Resumen

En este notebook aprendimos:

1. **Metricas de evaluacion**: Consistencia, longitud, cumplimiento de formato - herramientas
   cuantitativas para medir la calidad de los prompts
2. **A/B Testing**: Metodologia para comparar dos versiones de un prompt de manera objetiva
   usando evaluacion automatizada con LLM-as-judge
3. **Prompt Templates**: Plantillas parametrizables que estandarizan y hacen reutilizables
   nuestros prompts
4. **Optimizacion iterativa**: Proceso sistematico de mejora basado en metricas, pasando
   de un prompt basico a uno optimizado con evidencia cuantitativa

### Principios clave
- Siempre medir antes y despues de cada cambio
- Usar evaluacion automatizada para escalar las pruebas
- Documentar cada iteracion y los cambios realizados
- La optimizacion de prompts es un proceso cientifico, no artistico